In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from moc.configs.config import get_config
from moc.utils.run_config import RunConfig
from moc.datamodules.real_datamodule import RealDataModule
import torch
import pandas as pd
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
# from matplotlib.backends.backend_pdf import PdfPages

In [3]:
df = pd.read_csv("metrics-before-reg.csv")
config = get_config()
config.device = 'cuda'

In [4]:
def compute_null_hyp(nb_test_samples=1000, n_runs=10, n_points=100):
    pits = np.random.rand(nb_test_samples, n_runs, n_points)
    pits_tensor = torch.tensor(pits, dtype=torch.float32)
    sorted_pit = torch.sort(pits_tensor, dim=-1)[0]
    n = pits.shape[-1]
    lin = (torch.arange(n) + 1) / n
    calib = (sorted_pit - lin).abs().pow(1).mean(dim=-1).numpy()
    assert calib.ndim == 2
    return calib.mean(axis=-1)
 

In [5]:
pces_across_datasets = {}
test_statistics = {}
seeds = [0, 42, 866, 12, 4]
nb_test_samples= 1000
n_runs = 5
 
dataset_names = [
            ['camehl', 'households'], ['cevid', 'air'], 
            ['cevid', 'births1'],
            ['cevid', 'births2'], 
            ['cevid', 'wage'], 
            ['mulan', 'scm20d'],
            ['mulan', 'scm1d'], 
            ['mulan', 'wq'], ['mulan', 'scpf'], ['feldman', 'meps_21'], ['feldman', 'meps_19'],
            ['feldman', 'meps_20'], ['feldman', 'house'], ['feldman', 'bio'], ['feldman', 'blog_data'],
            ['del_barrio', 'calcofi'],
            ['del_barrio', 'ansur2'], ['wang', 'taxi']
            ]

In [6]:
def plot_pce_bar(ax, pces_real, test_statistics, stds_real=None, title=""):
    keys = list(test_statistics.keys())
    real_pces = [pces_real[k] for k in keys]
    real_stds = [stds_real[k] if stds_real is not None else 0 for k in keys]
 
    null_means = [test_statistics[k].mean() for k in keys]
    null_stds = [test_statistics[k].std() for k in keys]
 
    x = np.arange(len(keys))
    width = 0.8
 
    ax.bar(x, real_pces, width, label='MIX-NLL', yerr=real_stds, capsize=3, color='coral', zorder=1)
 
    ax.bar(x, null_means, width, label='Perfectly calibrated', yerr=null_stds, capsize=3, color='skyblue', zorder=2)
 
    ax.set_xticks(x)
    ax.set_xticklabels([f"{k[1]}" for k in keys], rotation=45, ha='right', fontsize=8)
    ax.set_ylabel("PCE")
    ax.set_title(title, fontsize=10)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

In [7]:
# Create a dictionnary of mapping
data_group_map = {data_name: data_group for data_group, data_name in dataset_names}
df = df[~df["data_name"].isin(["rf1", 'rf2', 'sf2'])]
# Add column "data_group"
df["data_group"] = df["data_name"].map(data_group_map)
 
 
# Need the key (data_group, data_name) for each line of df
df["key"] = df.apply(lambda row: (row["data_group"], row["data_name"]), axis=1)
 
grouped = df.groupby(["key", "prerank"])["pce"]

pces_real_all = grouped.mean().to_dict()
stds_real_all = grouped.std().to_dict()
 
# Generation of null distributions for each dataset
test_statistics = {}
for data_group, data_name in dataset_names:
    print(f"Working on dataset {data_name}")
    for seed in seeds:
        rc = RunConfig(config, data_group, data_name)
        datamodule = RealDataModule(rc, seed=seed)
        n = int(datamodule.total_size * datamodule.train_val_calib_test_split_ratio[-1])
        test_statistics[(data_group, data_name)] = compute_null_hyp(nb_test_samples, n_runs, n)
 

Working on dataset households


Working on dataset air
Working on dataset births1
Working on dataset births2
Working on dataset wage
Working on dataset scm20d
Working on dataset scm1d
Working on dataset wq
Working on dataset scpf
Working on dataset meps_21
Working on dataset meps_19
Working on dataset meps_20
Working on dataset house
Outlier present!
Outlier present!
Outlier present!
Outlier present!
Outlier present!
Working on dataset bio
Working on dataset blog_data
Working on dataset calcofi
Working on dataset ansur2
Working on dataset taxi


In [ ]:
# List of preranks
preranks = df["prerank"].unique()
 
 
# PDF creation 
# with PdfPages("pce_comparison_grid.pdf") as pdf:
fig = plt.figure(figsize=(14, 12))
gs = gridspec.GridSpec(4, 4)

for i, prerank in enumerate(preranks):
    selected_keys = [k for k in pces_real_all if k[1] == prerank]
    pces_real = {k[0]: pces_real_all[k] for k in selected_keys}
    stds_real = {k[0]: stds_real_all[k] for k in selected_keys}

    if i < 6:
        row = i // 2
        col = (i % 2) * 2
        ax = fig.add_subplot(gs[row, col:col+2])
    else:
        ax = fig.add_subplot(gs[3, 1:3])

    plot_pce_bar(ax, pces_real, test_statistics, stds_real, title=prerank.capitalize())

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=2, fontsize=10)
fig.tight_layout(rect=[0, 0, 1, 0.96])
# pdf.savefig(fig)
# plt.close(fig)

In [10]:
df.prerank.unique()

array(['marginal', 'mean', 'variance', 'dependency', 'pca', 'density',
       'cdf'], dtype=object)

In [12]:
custom_title = {
    "marginal": "Marginal",
    "mean": "Location",
    "variance": "Scale",
    "dependency": "Dependency",
    "pca": "PCA",
    "density": "HDR",
    "cdf": "Copula"
}

for prerank in df["prerank"].unique():
    selected_keys = [k for k in pces_real_all if k[1] == prerank]
    pces_real = {k[0]: pces_real_all[k] for k in selected_keys}
    stds_real = {k[0]: stds_real_all[k] for k in selected_keys}

    fig, ax = plt.subplots(figsize=(7, 4))
    plot_pce_bar(ax, pces_real, test_statistics, stds_real, title=custom_title[prerank])

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=8, loc='upper right')
    fig.tight_layout()
    fig.savefig(f"pce_{prerank}_hypothesis_test.png", bbox_inches='tight', dpi=500)
    plt.close(fig)

In [8]:
real_pce_means = {}
for key in pces_real_all:
    real_pce_means[key] = np.mean(pces_real_all[key])
 
null_pce_means = {}
for key in test_statistics:
    null_pce_means[key] = test_statistics[key]  

In [10]:
df.prerank.unique()

array(['marginal', 'mean', 'variance', 'dependency', 'pca', 'density',
       'cdf'], dtype=object)

In [ ]:
preranks = ['mean', 'variance', 'dependency', 'pca', 'density', 'cdf']
for prerank in preranks:
    keys_for_this_prerank = [k for k in pces_real_all if k[1] == prerank]
 
    num_plots = len(keys_for_this_prerank)
    cols = 6
    rows = (num_plots + cols - 1) // cols  
 
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
    axes = axes.flatten()
 
    for i, key in enumerate(keys_for_this_prerank):
        ax = axes[i]
        data_group, data_name = key[0]
        dataset_label = f"{data_name}"
        real_pce = real_pce_means[key]
        null_dist = null_pce_means[(data_group, data_name)]
 
        ax.hist(null_dist, bins=20, color='skyblue', edgecolor='skyblue')
        ax.axvline(real_pce, color='coral', linestyle='--', linewidth=3)
 
        ax.set_title(dataset_label, fontsize=17)
        ax.tick_params(axis='x', labelsize=13)
        ax.tick_params(axis='y', labelsize=13)
 
        ax.grid(axis='y', linestyle='--', alpha=0.5)
        ax.xaxis.set_major_locator(ticker.MaxNLocator(nbins=5))
 
        row_idx = i // cols
        col_idx = i % cols
 
        
        if row_idx == rows - 1:
            ax.set_xlabel("Test statistic", fontsize=17)
 
        
        if col_idx == 0:
            ax.set_ylabel("Count", fontsize=17)
 
    
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])
 
    
    # custom_line = plt.Line2D([0], [0], color='coral', linestyle='--', linewidth=2)
    # fig.legend([custom_line], ["MIX NLL"], loc='lower right', bbox_to_anchor=(0.7, 0.05), fontsize=20, ncol=1)
 
    
    plt.tight_layout()
    #plt.suptitle(f"Null Distribution of Mean PCE vs Real PCE ({prerank})", fontsize=30, y=1.03)
    plt.subplots_adjust(top=0.90, bottom=0.08)
 
    save_path = f"figures/pce_histograms_hyp_test/pce_null_vs_real_{prerank}.pdf"
    plt.savefig(save_path, bbox_inches='tight', dpi=500)
    plt.show()